In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pickle

with open('/content/drive/MyDrive/aqua spatial/enugu_data.pkl', 'rb') as f:
    data = pickle.load(f)

enugu = data['enugu']
wp_gdf = data['wp_gdf']
enugu_stats = data['enugu_stats']
lga_stats = data['lga_stats']

import pandas as pd
import numpy as np
import geopandas as gpd

gee_df = pd.read_csv('/content/drive/MyDrive/aqua spatial/gee_features_enugu.csv')

print(f" Notebook ready")
print(f"   LGAs: {len(enugu)}")
print(f"   Water points: {len(wp_gdf)}")
print(f"   GEE features: {gee_df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Notebook ready
   LGAs: 17
   Water points: 292
   GEE features: (17, 6)


Aggregating watter points to LGA level

In [18]:
# Each row = one LGA with summarized water point statistics
lga_water = wp_gdf.groupby('lga_name').agg(
    total_points        = ('status', 'count'),
    functional          = ('status', lambda x: (x == 'functional').sum()),
    non_functional      = ('status', lambda x: (x == 'non_functional').sum()),
    needs_repair        = ('status', lambda x: (x == 'needs_repair').sum()),
    avg_depth_m         = ('depth_m', 'mean'),
    max_depth_m         = ('depth_m', 'max'),
    total_pop_served    = ('population_served', 'sum'),
    avg_pop_per_point   = ('population_served', 'mean'),
    pct_contaminated    = ('water_quality', lambda x: (x == 'contaminated').mean() * 100),
    pct_acceptable      = ('water_quality', lambda x: (x == 'acceptable').mean() * 100),
    borehole_count      = ('source_type', lambda x: (x == 'borehole').sum()),
    handpump_count      = ('source_type', lambda x: (x == 'hand_pump').sum()),
).reset_index()

# Derived ratios — more meaningful than raw counts
lga_water['functionality_rate']   = (lga_water['functional'] / lga_water['total_points'] * 100).round(2)
lga_water['infrastructure_gap']   = (lga_water['non_functional'] / lga_water['total_points'] * 100).round(2)
lga_water['repair_burden']        = (lga_water['needs_repair'] / lga_water['total_points'] * 100).round(2)
lga_water['borehole_ratio']       = (lga_water['borehole_count'] / lga_water['total_points']).round(3)

print(" Water points aggregated to LGA level")
print(f"   Shape: {lga_water.shape}")
print(f"\nColumns: {lga_water.columns.tolist()}")
print(f"\nSample:")
print(lga_water.head(3).to_string(index=False))

 Water points aggregated to LGA level
   Shape: (17, 17)

Columns: ['lga_name', 'total_points', 'functional', 'non_functional', 'needs_repair', 'avg_depth_m', 'max_depth_m', 'total_pop_served', 'avg_pop_per_point', 'pct_contaminated', 'pct_acceptable', 'borehole_count', 'handpump_count', 'functionality_rate', 'infrastructure_gap', 'repair_burden', 'borehole_ratio']

Sample:
 lga_name  total_points  functional  non_functional  needs_repair  avg_depth_m  max_depth_m  total_pop_served  avg_pop_per_point  pct_contaminated  pct_acceptable  borehole_count  handpump_count  functionality_rate  infrastructure_gap  repair_burden  borehole_ratio
   Aninri            29          16               6             7    49.213793         77.3             11024         380.137931         24.137931        58.62069               7              13               55.17               20.69          24.14           0.241
     Awgu             8           5               2             1    51.875000         73.6

### LGA Area Calculations and Density Features


- Density features refers to where there are enough or most covered water points for good water infrastructure.

In [21]:
# Step 1: Calculate areas in UTM for accuracy
enugu_utm = enugu.to_crs(epsg=32632).copy()
enugu_utm['area_km2'] = (enugu_utm.geometry.area / 1e6).round(2)
lga_areas = enugu_utm[['NAME_2', 'area_km2']].rename(columns={'NAME_2': 'lga_name'})

# Step 2: Verify merge keys match before merging
missing = set(lga_water['lga_name']) - set(lga_areas['lga_name'])
if missing:
    print(f" LGAs in water data but not in boundary: {missing}")
else:
    print(" All LGA names match — merge is safe")

# Step 3: Merge
lga_water = lga_water.merge(lga_areas, on='lga_name', how='left')

# Step 4: Verify area_km2 exists before using it
assert 'area_km2' in lga_water.columns, " Merge failed — area_km2 missing"
assert lga_water['area_km2'].isna().sum() == 0, " Some LGAs missing area values"
print(f" Merge successful — area_km2 present for all {len(lga_water)} LGAs")

# Step 5: Density features
lga_water['point_density_per_km2']  = (lga_water['total_points'] / lga_water['area_km2']).round(4)
lga_water['pop_density_served_km2'] = (lga_water['total_pop_served'] / lga_water['area_km2']).round(2)

print(f"\n Final shape: {lga_water.shape}")
print(f"\nLGA areas (km²):")
print(lga_areas.sort_values('area_km2', ascending=False).to_string(index=False))
print(f"\nDensity ranking:")
print(lga_water[['lga_name','area_km2','point_density_per_km2','pop_density_served_km2']]
      .sort_values('point_density_per_km2', ascending=False)
      .to_string(index=False))

 All LGA names match — merge is safe
 Merge successful — area_km2 present for all 17 LGAs

 Final shape: (17, 22)

LGA areas (km²):
     lga_name  area_km2
          Udi    920.75
    Uzo-Uwani    898.66
      Isi-Uzo    890.11
    NkanuEast    826.87
       Ezeagu    616.09
       Nsukka    496.45
         Awgu    448.85
    Oji-River    371.65
   Igbo-Etiti    346.52
    EnuguEast    329.74
Igbo-ezeNorth    328.61
       Aninri    291.63
    NkanuWest    274.42
        Udenu    250.14
Igbo-ezeSouth    183.96
   EnuguNorth    157.35
   EnuguSouth     70.16

Density ranking:
     lga_name  area_km2  point_density_per_km2  pop_density_served_km2
    Oji-River    371.65                 0.0996                   36.95
       Aninri    291.63                 0.0994                   37.80
Igbo-ezeSouth    183.96                 0.0707                   35.02
        Udenu    250.14                 0.0680                   27.15
    Uzo-Uwani    898.66                 0.0501                 

Merging the water points features with the satelite features

In [24]:
# Step 1: Merge water features with GEE satellite features
df = lga_water.merge(gee_df, on='lga_name', how='left')

# Verify merge
assert len(df) == 17, f" Expected 17 rows, got {len(df)}"
assert df['ndvi'].isna().sum() == 0, " Missing GEE values after merge"
print(f" Master merge complete: {df.shape}")

# Step 2: Interaction features — combinations the model can't derive alone

# Drilling difficulty — deeper wells on steeper terrain = harder + costlier
df['drilling_difficulty'] = (df['avg_depth_m'] * df['slope']).round(3)

# Heat-vegetation stress index — high temp + low vegetation = water stressed
# Protect against division by zero
df['heat_veg_stress'] = (df['lst_celsius'] / df['ndvi'].replace(0, 0.001)).round(3)

# Soil-vegetation alignment — low soil moisture + low NDVI = doubly stressed
df['soil_veg_index'] = (df['soil_moisture'] * df['ndvi']).round(4)

# Infrastructure stress — non-functional + needs repair combined
df['total_stress_rate'] = (
    (df['non_functional'] + df['needs_repair']) / df['total_points'] * 100
).round(2)

# Terrain accessibility — high elevation + high slope = hard to reach
df['terrain_difficulty'] = (
    (df['elevation'] / df['elevation'].max()) * 0.5 +
    (df['slope'] / df['slope'].max()) * 0.5
).round(4)

# Step 3: Build composite water stress target variable
# This is what our model will learn to predict
# Higher score = more water stressed = higher priority for intervention

# Normalize each component to 0-1 scale
def normalize(series):
    """Min-max normalization — scales any feature to 0-1"""
    return (series - series.min()) / (series.max() - series.min())

df['target_stress_score'] = (
    normalize(df['total_stress_rate'])      * 0.30 +  # Infrastructure failure
    normalize(df['pct_contaminated'])       * 0.20 +  # Water quality
    normalize(df['heat_veg_stress'])        * 0.20 +  # Satellite stress signal
    normalize(df['terrain_difficulty'])     * 0.15 +  # Access difficulty
    normalize(1 / df['point_density_per_km2'].replace(0, 0.001)) * 0.15  # Coverage gap
).round(4)

print(f"\n Engineered features added: {df.shape}")
print(f"\n=== COMPOSITE WATER STRESS RANKINGS ===")
print(df[['lga_name', 'target_stress_score', 'drilling_difficulty',
          'heat_veg_stress', 'terrain_difficulty', 'total_stress_rate']]
      .sort_values('target_stress_score', ascending=False)
      .to_string(index=False))

 Master merge complete: (17, 27)

 Engineered features added: (17, 33)

=== COMPOSITE WATER STRESS RANKINGS ===
     lga_name  target_stress_score  drilling_difficulty  heat_veg_stress  terrain_difficulty  total_stress_rate
    EnuguEast               0.6030                3.265          182.908              0.4006             100.00
   EnuguSouth               0.5714               25.517          278.773              0.4358              50.00
Igbo-ezeSouth               0.5107                9.553          156.028              0.5499              69.23
   EnuguNorth               0.4922                9.729          278.773              0.4358              60.00
          Udi               0.3984               23.850          134.592              0.7381              47.62
   Igbo-Etiti               0.3812                9.481          106.804              0.6820              71.43
      Isi-Uzo               0.3427                6.797          143.516              0.3514            

In [26]:
import os
os.makedirs('/content/drive/MyDrive/aqua spatial', exist_ok=True)

# Save master dataset
df.to_csv('/content/drive/MyDrive/aqua spatial/master_features.csv', index=False)

# Save for pickle too
import pickle
with open('/content/drive/MyDrive/aqua spatial/master_features.pkl', 'wb') as f:
    pickle.dump(df, f)

print(" Master feature table saved")
print(f"   Rows: {len(df)} LGAs")
print(f"   Columns: {len(df.columns)} features")
print(f"\nFinal feature list:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:02d}. {col}")

 Master feature table saved
   Rows: 17 LGAs
   Columns: 33 features

Final feature list:
   01. lga_name
   02. total_points
   03. functional
   04. non_functional
   05. needs_repair
   06. avg_depth_m
   07. max_depth_m
   08. total_pop_served
   09. avg_pop_per_point
   10. pct_contaminated
   11. pct_acceptable
   12. borehole_count
   13. handpump_count
   14. functionality_rate
   15. infrastructure_gap
   16. repair_burden
   17. borehole_ratio
   18. area_km2_x
   19. point_density_per_km2
   20. pop_density_served_km2
   21. area_km2_y
   22. area_km2
   23. ndvi
   24. soil_moisture
   25. elevation
   26. slope
   27. lst_celsius
   28. drilling_difficulty
   29. heat_veg_stress
   30. soil_veg_index
   31. total_stress_rate
   32. terrain_difficulty
   33. target_stress_score
